# 02 · Experiments (Exp 0–6, baselines, age analyses)

**How to use:** *Runtime → Run all*. Nothing to edit. Allow Google Drive access when asked.

If Colab disconnects or the runtime ends, open the notebook again and *Run all* again: finished
work is kept on Drive and skipped, and the run continues where it stopped. All outputs,
checkpoints and caches are written to Drive (never to the temporary Colab disk).

**GPU:** this notebook is set to use a GPU (*Runtime → Change runtime type* shows T4). It also runs on
CPU, only slower.

Runs every experiment in `configs/run_plan.yaml` in priority order (primary models first), on the
graph run built by notebook 01. Each completed fold, inner-CV trial and permutation is saved on Drive
immediately, so the run can be continued across several Colab sessions until everything is done.
A failing experiment is recorded and the others continue. Cross-dataset settings run only if the
graph run passes the DESIGN.md §7 gate (otherwise they are listed as skipped).

In [ ]:
# ===== Setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_CODE = False   # leave False. True only if you were asked to switch to newer code
                      # (all notebooks then use the newest commit; unfinished work is recomputed).
BRANCH = 'claude/ecstatic-davinci-a2e8mh'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py'] + (['--update-code'] if UPDATE_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh
from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
print('outputs on Drive:', OUT)

## 1. Hardware and progress so far

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU only: slower)')
sh("python scripts/run_plan.py --paths configs/paths.yaml --status")

## 2. Run / continue all experiments

In [ ]:
try:
    sh("python scripts/run_plan.py --paths configs/paths.yaml")
except RuntimeError:
    print("\nWARNING: some experiments failed or are incompatible; see the table below (column message).")


## 3. Results so far (subject-level, mean over repeats)

In [ ]:
import pandas as pd
st = pd.read_csv(f"{OUT}/experiments/_plan_status.csv")
pd.set_option('display.max_rows', 500, 'display.width', 250)
st[['experiment', 'variant', 'setting', 'state', 'subject_roc_auc', 'subject_balanced_accuracy', 'message']]